# Compare Local Quantum SDK Simulators

## Problem

Run the same small benchmark workloads across installed local simulator SDKs and compare runtime, circuit depth, and output quality in a reproducible table.

## Quantum Advantage

None is claimed. These workloads are small enough for classical simulation; the goal is reproducible SDK comparison and result interpretation.

## SDK Advantages

- Cirq: simple local simulator and noise support in this package.
- PennyLane: local `default.qubit`/`default.mixed` devices and differentiable-programming ecosystem fit.
- Qiskit Aer: common transpilation-inclusive local simulator workflow.
- QuTiP: physics-oriented statevector simulation for small systems.

## Variables

- `candidate_backends`: local backends to try if installed.
- `active_backends`: installed local execution backends used in the comparison.
- `benchmarks`: GHZ and QFT benchmark definitions.
- `shots`: measurement samples per backend run.
- `repeats`: repeated executions used for local runtime statistics.


In [ ]:
import importlib.util

import matplotlib.pyplot as plt
import pandas as pd

from quantum_backend_bench.benchmarks.ghz import build_benchmark as build_ghz
from quantum_backend_bench.benchmarks.qft import build_benchmark as build_qft
from quantum_backend_bench.core.runner import run_benchmark
from quantum_backend_bench.utils.formatting import format_results_table
from quantum_backend_bench.utils.notebook import (
    check_runtime_samples,
    check_total_counts,
    notebook_artifact_dir,
    save_result_artifacts,
    verification_frame,
)

ARTIFACT_DIR = notebook_artifact_dir()

candidate_backends = {
    "cirq": ["cirq"],
    "pennylane": ["pennylane"],
    "qiskit_aer": ["qiskit", "qiskit_aer"],
    "qutip": ["qutip"],
}

active_backends = [
    backend
    for backend, modules in candidate_backends.items()
    if all(importlib.util.find_spec(module) is not None for module in modules)
]

shots = 256
repeats = 2
benchmarks = [
    build_ghz(n_qubits=4),
    build_qft(n_qubits=4),
]

print("Active local backends:", ", ".join(active_backends) if active_backends else "none")

## Run the Comparison

Each benchmark is built once and dispatched through the same package runner for every installed local backend.


In [ ]:
if not active_backends:
    raise RuntimeError(
        "No comparison backends are installed. Install at least quantum-backend-bench[cirq]."
    )

comparison_results = []
for benchmark in benchmarks:
    comparison_results.extend(
        run_benchmark(benchmark, active_backends, shots=shots, repeats=repeats)
    )

print(format_results_table(comparison_results))

comparison_table = pd.DataFrame(
    [
        {
            "case": result["metadata"]["case_label"],
            "benchmark": result["benchmark"],
            "backend": result["backend"],
            "runtime_seconds": result["metrics"]["runtime_seconds"],
            "runtime_stddev": result["metrics"]["runtime_seconds_stddev"],
            "depth": result["metrics"]["depth"],
            "gates": result["metrics"]["gate_count"],
            "two_qubit_gates": result["metrics"]["two_qubit_gate_count"],
            "tvd": result["metrics"]["total_variation_distance"],
        }
        for result in comparison_results
    ]
)
for column in ["runtime_seconds", "runtime_stddev", "tvd"]:
    comparison_table[column] = comparison_table[column].map(
        lambda value: None if value is None else round(value, 6)
    )
comparison_table

## Runtime and Structural Comparison

Runtime is local-machine dependent. Depth and gate counts are structural metrics and should be more stable across machines.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

runtime_pivot = comparison_table.pivot(
    index="backend", columns="benchmark", values="runtime_seconds"
)
runtime_pivot.plot(kind="bar", ax=axes[0])
axes[0].set_title("Runtime by backend")
axes[0].set_ylabel("seconds")
axes[0].tick_params(axis="x", rotation=30)

depth_pivot = comparison_table.pivot(index="backend", columns="benchmark", values="depth")
depth_pivot.plot(kind="bar", ax=axes[1])
axes[1].set_title("Circuit depth by backend")
axes[1].set_ylabel("depth")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Output Quality Snapshot

Total variation distance compares the measured distribution with the ideal distribution when the benchmark defines one. Lower is better, but shot count matters.


In [ ]:
quality_table = comparison_table[["case", "backend", "tvd"]].copy()
quality_table = quality_table.sort_values(["case", "tvd", "backend"], na_position="last")
quality_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
quality_plot = quality_table.dropna(subset=["tvd"]).pivot(
    index="backend", columns="case", values="tvd"
)
quality_plot.plot(kind="bar", ax=ax, color=["#2a9d8f", "#e76f51"])
ax.set_title("Total variation distance by backend")
ax.set_ylabel("TVD")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Save Reproducible Artifacts


In [ ]:
json_path, csv_path = save_result_artifacts(
    comparison_results, "local_simulator_comparison", ARTIFACT_DIR
)
print(f"Saved JSON: {json_path}")
print(f"Saved CSV: {csv_path}")

## Verification

The checks below confirm that every result contains the expected total shot count, runtime metadata is present, and each benchmark produced a consistent structural depth across local backends.


In [ ]:
checks = []
for result in comparison_results:
    total_check = check_total_counts(result, expected=shots * repeats)
    total_check["check"] = f"{result['metadata']['case_label']} / {result['backend']} total shots"
    checks.append(total_check)

    runtime_check = check_runtime_samples(result, expected=repeats)
    runtime_check["check"] = (
        f"{result['metadata']['case_label']} / {result['backend']} runtime samples"
    )
    checks.append(runtime_check)

for benchmark_name, group in comparison_table.groupby("benchmark"):
    depths = sorted(set(group["depth"]))
    checks.append(
        {
            "check": f"{benchmark_name} depth consistent across backends",
            "value": ", ".join(str(depth) for depth in depths),
            "expected": "one unique depth",
            "passed": len(depths) == 1,
        }
    )

verification = verification_frame(checks)
verification